# V6.6 — 5G Network Intelligence Final Reproducibility & Portfolio Audit

**Purpose:** final reproducibility audit and portfolio release build. Reconstruct the frozen source split, train source-only models, evaluate only on the previously inspected 21-session development partition, run an offline inference smoke test, and export auditable artifacts. This is not independent external validation. that ingests one 5G client-measurement CSV, engineers the established V5 features, forecasts downlink throughput at 5s/15s, scores near-term KPI-defined degradation risk, and creates persistent alert episodes.

**Evidence and scope**
- Training data: the uploaded UCC MISL / Raca-style client-side measurement traces.
- The warning target is a KPI-derived proxy, not an operator-confirmed fault.
- The model's `risk_score` is an uncalibrated model score, **not** a calibrated probability of network failure.
- The prototype is offline CSV inference, not live gNB/O-RAN integration or production monitoring.
- The previously inspected 21-session development holdout is not used to claim fresh validation. The deployable prototype models are fitted on the original 61-session training partition only.
- Rolling input features retain the V5 sample-window design; timestamps are used for 5s/15s targets and alert timing.

Run cells from top to bottom in Google Colab. The final cell launches an interactive upload dashboard.

In [ ]:
# 1. Environment and Google Drive
!pip -q install -U scikit-learn gradio

from google.colab import drive, files
drive.mount('/content/drive')

import os, re, io, glob, zipfile, json, warnings, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path('/content/drive/MyDrive/5G_Network_Intelligence_V6_6')
MODEL_DIR = PROJECT_DIR / 'models'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
for p in [PROJECT_DIR, MODEL_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_ZIP = Path('/content/drive/MyDrive/5G-production-dataset.zip')
if not DATA_ZIP.exists():
    print("Upload the original 5G-production-dataset.zip file.")
    uploaded = files.upload()
    zip_candidates = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_candidates:
        raise FileNotFoundError("No ZIP uploaded. Please upload 5G-production-dataset.zip.")
    DATA_ZIP = zip_candidates[0]

print("Dataset archive:", DATA_ZIP)
print("Artifacts will be saved to:", PROJECT_DIR)

## 2. Load and standardize the original dataset

The parser follows the V5 column aliases and unit conversion. It retains source-relative CSV paths as session identifiers so the original group split can be reconstructed.

In [ ]:
ALIASES = {
    'ts':['timestamp','time'], 'dl':['dl_bitrate','download','dl_throughput'],
    'ul':['ul_bitrate','upload','ul_throughput'], 'rsrp':['rsrp'], 'rsrq':['rsrq'],
    'snr':['snr','sinr'], 'cqi':['cqi'], 'rssi':['rssi'],
    'cell':['cellid','cell_id'], 'speed':['speed'], 'lat':['latitude'], 'lon':['longitude'],
    'ping':['pingavg','ping_avg','rtt','latency'], 'loss':['pingloss','packet_loss'],
}
NUM = ['dl','ul','rsrp','rsrq','snr','cqi','rssi','cell','speed','lat','lon','ping','loss']
BASE = ['dl_mbps','ul_mbps','rsrp','rsrq','snr','cqi','rssi','speed','ping','loss']

def standardize(raw, session_id):
    raw = raw.copy()
    raw.columns = [str(c).strip().lower() for c in raw.columns]
    out = pd.DataFrame(index=raw.index)
    for key, candidates in ALIASES.items():
        col = next((c for c in candidates if c in raw.columns), None)
        out[key] = raw[col] if col else np.nan
    ts = pd.to_datetime(out['ts'], format='%Y.%m.%d_%H.%M.%S', errors='coerce')
    if ts.isna().all():
        ts = pd.to_datetime(out['ts'], errors='coerce')
    out['ts'] = ts
    for c in NUM:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=['ts']).sort_values('ts')
    out['session'] = str(session_id)
    out['app'] = os.path.basename(os.path.dirname(str(session_id)))
    out['dl_mbps'] = out['dl'] / 1000.0
    out['ul_mbps'] = out['ul'] / 1000.0
    return out.reset_index(drop=True)

def read_training_archive(zip_path):
    frames = []
    with zipfile.ZipFile(zip_path) as z:
        names = sorted(n for n in z.namelist()
                       if n.lower().endswith('.csv') and '5g-production-dataset/' in n.lower() and '__macosx/' not in n.lower() and not Path(n).name.startswith('._'))
        if not names:
            names = sorted(n for n in z.namelist() if n.lower().endswith('.csv') and '__macosx/' not in n.lower() and not Path(n).name.startswith('._'))
        for name in names:
            try:
                raw = pd.read_csv(io.BytesIO(z.read(name)),
                                  na_values=['-',' -','','NaN'], skipinitialspace=True)
                s = standardize(raw, name)
                if len(s) > 60 and s['dl'].notna().sum() > 30:
                    frames.append(s)
            except Exception as e:
                print("Skipped", name, ":", str(e)[:160])
    if not frames:
        raise ValueError("No usable CSV sessions found in the archive.")
    return pd.concat(frames, ignore_index=True)

df_raw = read_training_archive(DATA_ZIP)
print("Raw rows:", len(df_raw), "| sessions:", df_raw.session.nunique())
display(df_raw.head())

## 3. Rebuild the V5 feature pipeline and original session split

Duplicate timestamps are aggregated by median for continuous KPIs and last value for identifiers. Feature definitions match the V5 pipeline. The original 61/21 session split is reconstructed from the pre-deduplicated feature matrix; only the 61 training sessions are used to fit the prototype models.

In [ ]:
def make_features(g):
    g = g.sort_values('ts').reset_index(drop=True).copy()
    g['ho'] = ((g['cell'] != g['cell'].shift()) &
               g['cell'].notna() & g['cell'].shift().notna()).astype(float)
    cols = BASE + ['ho']
    g = g.set_index('ts')[cols + ['session','app']]
    g[BASE] = g[BASE].interpolate(limit=5)
    g = g.dropna(subset=['dl_mbps','rsrp','snr']).reset_index()
    for w in (10, 30):
        g[f'dl_m{w}'] = g['dl_mbps'].rolling(w, min_periods=3).mean()
        g[f'dl_s{w}'] = g['dl_mbps'].rolling(w, min_periods=3).std()
        g[f'rsrp_m{w}'] = g['rsrp'].rolling(w, min_periods=3).mean()
        g[f'rsrp_s{w}'] = g['rsrp'].rolling(w, min_periods=3).std()
        g[f'snr_m{w}'] = g['snr'].rolling(w, min_periods=3).mean()
        g[f'ho_sum{w}'] = g['ho'].rolling(w, min_periods=1).sum()
    for L in (1,2,3,5,10,15,30):
        g[f'dl_lag{L}'] = g['dl_mbps'].shift(L)
    g['dl_chg'] = g['dl_mbps'].diff()
    g['rsrp_chg'] = g['rsrp'].diff()
    med60 = g['dl_mbps'].rolling(60, min_periods=10).median().shift(1)
    g['dl_vs_med60'] = g['dl_mbps'] / (med60 + 1e-6)
    return g

# Pre-dedup features reproduce the original deterministic session partition.
feat_original = pd.concat(
    [make_features(g) for _, g in df_raw.groupby('session', sort=False)],
    ignore_index=True
)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(feat_original, groups=feat_original['session']))
train_original = feat_original.iloc[tr_idx].copy()
train_sessions = set(train_original['session'].unique())
development_sessions = set(feat_original.iloc[te_idx]['session'].unique())

# Deduplicate timestamps exactly as the V5 elapsed-time pipeline.
continuous_cols = ['dl','ul','rsrp','rsrq','snr','cqi','rssi','speed','lat','lon','ping','loss']
cleaned = []
for sid, g in df_raw.groupby('session', sort=False):
    agg = {c:'median' for c in continuous_cols if c in g.columns}
    for c in ['cell','app','session']:
        if c in g.columns:
            agg[c] = 'last'
    cleaned.append(g.sort_values('ts').groupby('ts', as_index=False).agg(agg))
df_clean = pd.concat(cleaned, ignore_index=True).sort_values(['session','ts'])
df_clean['dl_mbps'] = df_clean['dl'] / 1000.0
df_clean['ul_mbps'] = df_clean['ul'] / 1000.0

feat = pd.concat([make_features(g) for _, g in df_clean.groupby('session', sort=False)],
                 ignore_index=True)
feat = feat.sort_values(['session','ts']).reset_index(drop=True)

FEATS = [c for c in feat_original.columns
         if c not in ('ts','session','app','degraded','y_event') and not c.startswith('y_dl')]
FEATS = [c for c in FEATS if c in feat.columns and
         train_original[c].notna().sum() > 50 and train_original[c].nunique() > 1]
FEATS_V5 = FEATS.copy()

# Frozen cohort checks: the exact pre-dedup eligibility and seed must reproduce 61/21.
assert len(train_sessions) == 61, f'Expected 61 training sessions, got {len(train_sessions)}'
assert len(development_sessions) == 21, f'Expected 21 development sessions, got {len(development_sessions)}'
assert not (train_sessions & development_sessions)
excluded_hsdpa = '5G-production-dataset/Download/Driving/B_2019.12.16_11.49.59.csv'
assert excluded_hsdpa not in train_sessions and excluded_hsdpa not in development_sessions

print("Feature rows:", len(feat), "| predictors:", len(FEATS_V5))
print("Training sessions:", len(train_sessions),
      "| previously inspected development sessions:", len(development_sessions))
print("Training session overlap check:", len(train_sessions & development_sessions))
assert not (train_sessions & development_sessions)
display(pd.DataFrame({'feature':FEATS_V5}))

## 4. Create elapsed-time forecasting targets and KPI-defined warning labels

Forecast targets use the first observation at or after 5s/15s, allowing at most 2.5s overshoot. The warning label is a proxy: downlink below 5 Mbps together with RSRP below -110 dBm or SNR below 0 dB, sustained for at least 3 elapsed seconds. A row is labelled positive when such an onset occurs within the next 10 seconds.

In [ ]:
# Elapsed-time throughput targets
for h in [5, 15]:
    feat[f'y_dl_after_{h}s'] = np.nan

for sid, idx in feat.groupby('session', sort=False).groups.items():
    g = feat.loc[list(idx)].sort_values('ts')
    pos = g.index.to_numpy()
    times = pd.to_datetime(g['ts']).astype('int64').to_numpy() / 1e9
    values = g['dl_mbps'].to_numpy(dtype=float)
    for h in [5, 15]:
        j = np.searchsorted(times, times + h, side='left')
        valid = j < len(times)
        rows = np.where(valid)[0]
        if len(rows):
            keep = (times[j[rows]] - (times[rows] + h)) <= 2.5
            src = rows[keep]
            feat.loc[pos[src], f'y_dl_after_{h}s'] = values[j[src]]

DL_EVENT_THR = 5.0
RSRP_EVENT_THR = -110.0
SNR_EVENT_THR = 0.0
MIN_EVENT_DURATION_S = 3.0
MAX_GAP_S = 2.5
WARNING_HORIZON_S = 10.0

def build_event_arrays(g):
    g = g.sort_values('ts').reset_index(drop=True)
    t = pd.to_datetime(g['ts']).astype('int64').to_numpy() / 1e9
    radio = ((g['rsrp'] < RSRP_EVENT_THR) | (g['snr'] < SNR_EVENT_THR)).fillna(False).to_numpy()
    candidate = (g['dl_mbps'] < DL_EVENT_THR).fillna(False).to_numpy() & radio
    sustained = np.zeros(len(g), dtype=bool)
    onset = np.zeros(len(g), dtype=bool)
    start = 0
    while start < len(g):
        if not candidate[start]:
            start += 1
            continue
        end = start + 1
        while end < len(g) and candidate[end] and (t[end]-t[end-1]) <= MAX_GAP_S:
            end += 1
        duration = t[end-1] - t[start] if end-start > 1 else 0.0
        if end-start > 1 and duration >= MIN_EVENT_DURATION_S:
            sustained[start:end] = True
            onset[start] = True
        start = end
    return t, sustained, onset

labelled_parts = []
for sid, g in feat.groupby('session', sort=False):
    g = g.sort_values('ts').copy().reset_index(drop=True)
    t, sustained, onset = build_event_arrays(g)
    y = np.full(len(g), np.nan, dtype=float)
    last_t = t[-1] if len(t) else np.nan
    for i in range(len(g)):
        if sustained[i] or t[i] + WARNING_HORIZON_S > last_t:
            continue
        j0 = np.searchsorted(t, t[i], side='right')
        j1 = np.searchsorted(t, t[i] + WARNING_HORIZON_S, side='right')
        y[i] = float(onset[j0:j1].any())
    g['v5_event'] = sustained.astype(int)
    g['v5_event_onset'] = onset.astype(int)
    g['v5_y_event_onset'] = y
    labelled_parts.append(g)
feat_labelled = pd.concat(labelled_parts, ignore_index=True)

print("Proxy event onsets:", int(feat_labelled.v5_event_onset.sum()))
print("Warning target prevalence (all sessions, descriptive):",
      round(feat_labelled.v5_y_event_onset.mean(), 4))
print("Training sessions available:", feat_labelled.session.isin(train_sessions).sum(), "rows")

## 5. Fit the prototype models on the original training sessions only

Two throughput regressors and one degradation-risk classifier are fitted. Sample weights give each session equal total weight. The classifier's output is called a **risk score** because its probability calibration has not been established across domains.

In [ ]:
def session_balanced_weights(frame):
    counts = frame.groupby('session')['session'].transform('size').astype(float)
    w = 1.0 / counts
    return (w / w.mean()).to_numpy()

train_frame = feat_labelled[feat_labelled.session.isin(train_sessions)].copy()
models = {}
training_summary = []

for h in [5, 15]:
    target = f'y_dl_after_{h}s'
    fit = train_frame.dropna(subset=[target]).copy()
    if fit.session.nunique() < 5:
        raise RuntimeError(f"Insufficient training sessions for {h}s forecast.")
    model = HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_depth=6,
        l2_regularization=1.0, random_state=RANDOM_STATE
    )
    model.fit(fit[FEATS_V5], fit[target],
              sample_weight=session_balanced_weights(fit))
    models[f'forecast_{h}s'] = model
    joblib.dump(model, MODEL_DIR / f'forecast_{h}s.joblib')
    training_summary.append({
        'model':f'forecast_{h}s','training_sessions':fit.session.nunique(),
        'training_rows':len(fit),'target_rows':fit[target].notna().sum()
    })

clf_fit = train_frame.dropna(subset=['v5_y_event_onset']).copy()
clf_fit['v5_y_event_onset'] = clf_fit['v5_y_event_onset'].astype(int)
if clf_fit.session.nunique() < 5 or clf_fit.v5_y_event_onset.nunique() != 2:
    raise RuntimeError("Warning model requires at least five sessions and both target classes.")
risk_model = HistGradientBoostingClassifier(
    max_iter=350, learning_rate=0.04, max_depth=4,
    l2_regularization=2.0, class_weight='balanced',
    random_state=RANDOM_STATE
)
risk_model.fit(clf_fit[FEATS_V5], clf_fit['v5_y_event_onset'],
               sample_weight=session_balanced_weights(clf_fit))
models['risk'] = risk_model
joblib.dump(risk_model, MODEL_DIR / 'degradation_risk_model.joblib')

# Persist exact feature order and model configuration with the models.
metadata = {
    'prototype_version':'V6.6',
    'feature_names':FEATS_V5,
    'training_sessions':sorted(train_sessions),
    'training_session_count':len(train_sessions),
    'warning_label':'KPI-derived proxy: DL<5 Mbps and (RSRP<-110 dBm or SNR<0 dB), sustained >=3 seconds; onset within next 10 seconds',
    'risk_score_note':'Uncalibrated model score; not a calibrated incident probability',
    'forecast_horizons_seconds':[5,15],
    'max_timestamp_gap_seconds':MAX_GAP_S,
    'rolling_features':'Sample-window based, inherited from V5'
}
with open(PROJECT_DIR / 'model_metadata.json','w') as f:
    json.dump(metadata,f,indent=2)
pd.DataFrame(training_summary).to_csv(OUTPUT_DIR / 'training_build_summary.csv',index=False)

print("Models fitted and saved.")
display(pd.DataFrame(training_summary))
print("Model directory:", MODEL_DIR)

## 6A. Frozen development evaluation (previously inspected; not external validation)

This evaluates the fitted source-only models on the 21-session development partition. The partition was inspected in earlier V5/V6 iterations, so metrics are development evidence, not an untouched final test. Warning labels are KPI-derived proxy labels, not operator-confirmed incidents.


In [ ]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    confusion_matrix)

# Forecast evaluation: same development sessions, elapsed-time targets, persistence baseline.
dev_frame = feat_labelled[feat_labelled.session.isin(development_sessions)].copy()
forecast_rows=[]; per_session_rows=[]; forecast_predictions=[]
for h in [5,15]:
    target=f'y_dl_after_{h}s'
    q=dev_frame.dropna(subset=[target]).copy()
    q=q.loc[q[FEATS_V5].notna().any(axis=1)].copy()
    q['prediction_mbps_raw']=models[f'forecast_{h}s'].predict(q[FEATS_V5])
    q['prediction_mbps']=np.maximum(q['prediction_mbps_raw'],0.0)
    q['persistence_mbps']=q['dl_mbps']
    q['horizon_s']=h
    q['model_abs_error']=(q[target]-q['prediction_mbps']).abs()
    q['persistence_abs_error']=(q[target]-q['persistence_mbps']).abs()
    y=q[target].to_numpy(); p=q['prediction_mbps'].to_numpy(); b=q['persistence_mbps'].to_numpy()
    forecast_rows.append({'horizon_s':h,'n_rows':len(q),'n_sessions':q.session.nunique(),
        'model_MAE_Mbps':mean_absolute_error(y,p),'persistence_MAE_Mbps':mean_absolute_error(y,b),
        'MAE_improvement_pct':(mean_absolute_error(y,b)-mean_absolute_error(y,p))/mean_absolute_error(y,b)*100,
        'model_RMSE_Mbps':mean_squared_error(y,p)**0.5,'persistence_RMSE_Mbps':mean_squared_error(y,b)**0.5,
        'model_R2':r2_score(y,p),'negative_raw_prediction_count':int((q.prediction_mbps_raw<0).sum()),
        'evaluation_partition':'previously inspected development sessions'})
    for sid,g in q.groupby('session'):
        my=mean_absolute_error(g[target],g.prediction_mbps)
        by=mean_absolute_error(g[target],g.persistence_mbps)
        per_session_rows.append({'session':sid,'horizon_s':h,'rows':len(g),'model_MAE_Mbps':my,
            'persistence_MAE_Mbps':by,'MAE_improvement_pct':(by-my)/by*100 if by else np.nan})
    forecast_predictions.append(q[['session','ts',target,'dl_mbps','prediction_mbps_raw','prediction_mbps','persistence_mbps','horizon_s']])

forecast_summary=pd.DataFrame(forecast_rows)
per_session_forecast=pd.DataFrame(per_session_rows)
forecast_predictions=pd.concat(forecast_predictions,ignore_index=True)

# Session-cluster bootstrap over the already inspected development sessions; descriptive only.
rng=np.random.default_rng(RANDOM_STATE)
boot_rows=[]
for h,g in per_session_forecast.groupby('horizon_s'):
    vals=g.MAE_improvement_pct.dropna().to_numpy()
    samples=[]
    for _ in range(2000):
        z=rng.choice(vals,size=len(vals),replace=True)
        samples.append(np.mean(z))
    boot_rows.append({'horizon_s':int(h),'sessions':len(vals),'mean_session_improvement_pct':float(np.mean(vals)),
        'ci95_low_pct':float(np.quantile(samples,.025)),'ci95_high_pct':float(np.quantile(samples,.975)),
        'bootstrap_unit':'session','interpretation':'descriptive development-set uncertainty; not independent external validation'})
forecast_bootstrap=pd.DataFrame(boot_rows)

# Warning model evaluation. This is a proxy-label discrimination audit, not incident prediction validation.
warn=dev_frame.dropna(subset=['v5_y_event_onset']).copy()
warn['y_true']=warn['v5_y_event_onset'].astype(int)
warn['risk_score']=models['risk'].predict_proba(warn[FEATS_V5])[:,1]
warn['predicted_at_050']= (warn.risk_score>=0.50).astype(int)
yw=warn.y_true.to_numpy(); sw=warn.risk_score.to_numpy(); pw=warn.predicted_at_050.to_numpy()
warning_metrics=pd.DataFrame([{
    'n_rows':len(warn),'n_sessions':warn.session.nunique(),'proxy_prevalence':float(np.mean(yw)),
    'ROC_AUC':roc_auc_score(yw,sw) if len(np.unique(yw))==2 else np.nan,
    'PR_AUC_average_precision':average_precision_score(yw,sw) if len(np.unique(yw))==2 else np.nan,
    'Brier_score':brier_score_loss(yw,sw),'precision_at_050':precision_score(yw,pw,zero_division=0),
    'recall_at_050':recall_score(yw,pw,zero_division=0),'F1_at_050':f1_score(yw,pw,zero_division=0),
    'balanced_accuracy_at_050':balanced_accuracy_score(yw,pw),
    'threshold':0.50,'score_calibrated_probability':False,
    'label_type':'KPI-derived proxy; not operator-confirmed incidents'}])
cm=confusion_matrix(yw,pw,labels=[0,1])
warning_confusion=pd.DataFrame(cm,index=['true_0','true_1'],columns=['pred_0','pred_1']).reset_index(names='actual_class')

# Session-level warning metrics to expose heterogeneity.
warning_session_rows=[]
for sid,g in warn.groupby('session'):
    yy=g.y_true.to_numpy(); ss=g.risk_score.to_numpy(); pp=(ss>=.50).astype(int)
    warning_session_rows.append({'session':sid,'rows':len(g),'prevalence':float(yy.mean()),
        'precision_at_050':precision_score(yy,pp,zero_division=0),
        'recall_at_050':recall_score(yy,pp,zero_division=0),
        'F1_at_050':f1_score(yy,pp,zero_division=0)})
warning_session_metrics=pd.DataFrame(warning_session_rows)

print('Forecast summary'); display(forecast_summary)
print('Session-cluster bootstrap (descriptive)'); display(forecast_bootstrap)
print('Warning proxy metrics'); display(warning_metrics)
print('Confusion matrix'); display(warning_confusion)


## 7. Inference, persistent alert engine and KPI-based explanation hints

The inference path processes one uploaded CSV. Alerts are emitted only after K consecutive above-threshold scores within the allowed timestamp gap, with a cooldown. Threshold is a user-controlled operating setting—not a validated optimum. Explanations are descriptive KPI patterns, not confirmed root causes.

In [ ]:
def build_alerts(pred_frame, threshold=0.50, persistence_k=2, cooldown_s=10.0):
    episodes = []
    for sid, g in pred_frame.groupby('session', sort=False):
        g = g.sort_values('ts').reset_index(drop=True)
        times = pd.to_datetime(g.ts).tolist()
        scores = g.risk_score.to_numpy(float)
        flags = scores >= float(threshold)
        i, last_alert = 0, None
        while i < len(g):
            if not flags[i]:
                i += 1
                continue
            start, end = i, i + 1
            while end < len(g) and flags[end]:
                gap = (times[end] - times[end-1]).total_seconds()
                if gap <= 0 or gap > MAX_GAP_S:
                    break
                end += 1
            if end-start >= int(persistence_k):
                confirm = start + int(persistence_k) - 1
                alert_time = times[confirm]
                cooldown_ok = last_alert is None or (alert_time-last_alert).total_seconds() >= cooldown_s
                if cooldown_ok:
                    row = g.iloc[confirm]
                    reasons = []
                    if row.get('dl_mbps', np.nan) < DL_EVENT_THR:
                        reasons.append('Current downlink below 5 Mbps')
                    if row.get('rsrp', np.nan) < RSRP_EVENT_THR:
                        reasons.append('Weak RSRP indicator')
                    if row.get('snr', np.nan) < SNR_EVENT_THR:
                        reasons.append('Low SNR indicator')
                    if row.get('ho_sum10', 0) > 0:
                        reasons.append('Recent handover activity')
                    if pd.isna(row.get('ping', np.nan)):
                        reasons.append('Latency unavailable in source data')
                    if pd.isna(row.get('loss', np.nan)):
                        reasons.append('Packet-loss unavailable in source data')
                    if not reasons:
                        reasons.append('Elevated model score; inspect underlying KPIs')
                    episodes.append({
                        'session':sid,'alert_time':alert_time,
                        'risk_score_at_confirmation':float(scores[confirm]),
                        'persistence_rows':int(persistence_k),
                        'confirmation_delay_seconds':float((times[confirm]-times[start]).total_seconds()),
                        'kpi_explanation_hints':'; '.join(reasons),
                        'interpretation':'KPI-based hypotheses only; not a verified root cause'
                    })
                    last_alert = alert_time
            i = end
    return pd.DataFrame(episodes)

def run_inference(raw_csv, threshold=0.50, persistence_k=2, cooldown_s=10.0):
    raw = pd.read_csv(raw_csv, na_values=['-',' -','','NaN'], skipinitialspace=True)
    sid = Path(raw_csv).name
    g = standardize(raw, sid)
    if len(g) < 35:
        raise ValueError("CSV has too few valid timestamped rows for rolling features.")
    # Deduplicate timestamps within this uploaded session.
    agg = {c:'median' for c in continuous_cols if c in g.columns}
    for c in ['cell','app','session']:
        if c in g.columns:
            agg[c] = 'last'
    g = g.groupby('ts',as_index=False).agg(agg).sort_values('ts')
    g['dl_mbps'] = g['dl']/1000.0
    g['ul_mbps'] = g['ul']/1000.0
    x = make_features(g)
    if x.empty:
        raise ValueError("No rows remain after timestamp and required KPI validation.")
    # Ensure the exact training feature schema and order.
    for c in FEATS_V5:
        if c not in x.columns:
            x[c] = np.nan
    X = x[FEATS_V5]
    x['forecast_5s_mbps'] = np.maximum(models['forecast_5s'].predict(X), 0.0)
    x['forecast_15s_mbps'] = np.maximum(models['forecast_15s'].predict(X), 0.0)
    x['risk_score'] = models['risk'].predict_proba(X)[:,1]
    x['risk_threshold'] = float(threshold)
    x['threshold_positive'] = (x.risk_score >= float(threshold)).astype(int)
    alerts = build_alerts(x, threshold, persistence_k, cooldown_s)

    output_cols = ['ts','dl_mbps','rsrp','snr','ping','loss','speed',
                   'forecast_5s_mbps','forecast_15s_mbps','risk_score',
                   'threshold_positive']
    result = x[[c for c in output_cols if c in x.columns]].copy()
    result['alert_confirmed'] = 0
    if not alerts.empty:
        alert_times = set(pd.to_datetime(alerts.alert_time))
        result.loc[pd.to_datetime(result.ts).isin(alert_times),'alert_confirmed'] = 1

    summary = {
        'session_file':sid,
        'valid_rows_scored':int(len(result)),
        'mean_current_downlink_mbps':float(result.dl_mbps.mean()),
        'mean_forecast_5s_mbps':float(result.forecast_5s_mbps.mean()),
        'mean_forecast_15s_mbps':float(result.forecast_15s_mbps.mean()),
        'max_risk_score':float(result.risk_score.max()),
        'rows_above_threshold':int(result.threshold_positive.sum()),
        'confirmed_alert_episodes':int(len(alerts)),
        'risk_score_is_calibrated_probability':False,
        'data_mode':'Offline uploaded CSV',
        'root_cause_status':'KPI hypotheses only; no operator-confirmed cause'
    }
    return result, alerts, summary

print("Inference and alert functions are ready.")

## 8. Test the inference pipeline on one original session (functional smoke test)

This is a software smoke test on an existing source session, not an independent model-performance evaluation. It checks that the complete pipeline runs end-to-end.

In [ ]:
# Export one original-format session CSV from the ZIP as a software smoke test.
with zipfile.ZipFile(DATA_ZIP) as z:
    source_names = sorted(
        n for n in z.namelist()
        if n.lower().endswith('.csv') and '__macosx/' not in n.lower()
        and not Path(n).name.startswith('._')
        and '5g-production-dataset/' in n.lower()
    )
    if not source_names:
        raise FileNotFoundError("No original-format session CSV found in the ZIP.")
    smoke_name = source_names[0]
    smoke_raw = pd.read_csv(io.BytesIO(z.read(smoke_name)),
                            na_values=['-',' -','','NaN'], skipinitialspace=True)

smoke_path = OUTPUT_DIR / 'smoke_test_original_format.csv'
smoke_raw.to_csv(smoke_path,index=False)
smoke_result, smoke_alerts, smoke_summary = run_inference(
    str(smoke_path), threshold=0.50, persistence_k=2, cooldown_s=10.0
)
print(json.dumps(smoke_summary,indent=2,default=str))
display(smoke_result.head(10))
display(smoke_alerts.head(10))
smoke_result.to_csv(OUTPUT_DIR / 'smoke_test_predictions.csv',index=False)
smoke_alerts.to_csv(OUTPUT_DIR / 'smoke_test_alerts.csv',index=False)
print("Smoke test completed. This confirms execution, not predictive validity.")

## 9A. Export final reproducibility and portfolio artifacts

This cell writes the source manifests, forecast and warning metrics, model files, smoke-test outputs, hashes and limitations report. It does not change the frozen models or tune on NordicDat.


In [ ]:
# Save all audit outputs and reproducibility artifacts.
manifest = pd.DataFrame([{'session':s,'partition':'train'} for s in sorted(train_sessions)] +
                        [{'session':s,'partition':'development_previously_inspected_not_external'} for s in sorted(development_sessions)])
manifest.to_csv(OUTPUT_DIR/'v6_6_source_session_manifest.csv',index=False)
forecast_summary.to_csv(OUTPUT_DIR/'v6_6_forecast_summary.csv',index=False)
per_session_forecast.to_csv(OUTPUT_DIR/'v6_6_per_session_forecast.csv',index=False)
forecast_predictions.to_csv(OUTPUT_DIR/'v6_6_forecast_predictions.csv',index=False)
forecast_bootstrap.to_csv(OUTPUT_DIR/'v6_6_forecast_session_bootstrap_descriptive.csv',index=False)
warning_metrics.to_csv(OUTPUT_DIR/'v6_6_warning_proxy_metrics.csv',index=False)
warning_confusion.to_csv(OUTPUT_DIR/'v6_6_warning_confusion_matrix.csv',index=False)
warning_session_metrics.to_csv(OUTPUT_DIR/'v6_6_warning_session_metrics.csv',index=False)

# Archive fingerprint and feature/model metadata.
archive_sha256=hashlib.sha256(DATA_ZIP.read_bytes()).hexdigest()
release_metadata={
 'version':'V6.6','source_archive_sha256':archive_sha256,
 'source_csv_files_in_archive':int(len(source_names)) if 'source_names' in globals() else None,
 'eligible_source_sessions':int(len(train_sessions)+len(development_sessions)),
 'train_sessions':int(len(train_sessions)),'development_sessions_previously_inspected':int(len(development_sessions)),
 'split_seed':RANDOM_STATE,'test_size':0.25,'forecast_horizons_seconds':[5,15],
 'max_target_overshoot_seconds':2.5,'feature_names':FEATS_V5,
 'warning_label':'KPI-derived proxy: DL<5 Mbps and (RSRP<-110 dBm or SNR<0 dB), sustained >=3 seconds; onset within next 10 seconds',
 'warning_score_calibrated_probability':False,
 'evaluation_note':'21-session partition was inspected in prior V5/V6 development; not untouched external validation',
 'external_transfer_note':'NordicDat V6.5.1 transfer underperformed persistence; result retained as documented domain-transfer failure; no NordicDat tuning performed in V6.6',
 'telemetry_mode':'offline uploaded CSV; not live gNB/operator telemetry',
 'root_cause_status':'KPI-based hypotheses only; no operator-confirmed root cause',
 'incident_labels':'KPI-derived proxy labels; no operator-confirmed incident labels',
 'rolling_features':'sample-window based, not guaranteed elapsed-time windows',
 'model_type':'HistGradientBoostingRegressor x2; HistGradientBoostingClassifier x1',
 'prediction_nonnegative_in_inference':True,
 'prediction_policy':'clip forecast values at zero; raw inference values are not exported in the dashboard path'
}
(OUTPUT_DIR/'v6_6_release_metadata.json').write_text(json.dumps(release_metadata,indent=2))

report=f"""# 5G Network Intelligence — V6.6 Final Audit Report\n\n## Scope\nOffline CSV prototype using client-side 5G measurement traces. This is not live operator telemetry or production network integration.\n\n## Reproducibility\n- Source archive SHA-256: `{archive_sha256}`\n- Frozen split: {len(train_sessions)} training sessions / {len(development_sessions)} previously inspected development sessions\n- Forecast horizons: 5s and 15s; maximum target overshoot 2.5s\n- Models: two HistGradientBoosting regressors and one HistGradientBoosting classifier\n\n## Forecast results\n{forecast_summary.to_markdown(index=False)}\n\n## Warning proxy results\n{warning_metrics.to_markdown(index=False)}\n\nWarning scores are uncalibrated. Labels are KPI-derived proxies, not operator-confirmed incidents.\n\n## External transfer\nNordicDat V6.5.1 evaluation underperformed persistence. This result is retained as a documented domain-transfer failure. NordicDat was not used for model fitting or tuning in V6.6.\n\n## Limitations\n- Development sessions were inspected in earlier iterations; these are not untouched final test data.\n- Original dataset contains client-side measurement traces, not operator gNB counters.\n- No CPU/memory counters, operator incident labels, or confirmed root causes.\n- Alert explanations are KPI-based hypotheses only.\n- Rolling features are sample-window based.\n- Offline CSV inference only; no Kafka, O-RAN, gNB, or live operator integration.\n- Warning score is not a calibrated probability.\n\n## Release decision\nThis is a reproducible research/portfolio prototype. Do not describe it as production-ready or as validated operator-grade failure prediction.\n"""
(OUTPUT_DIR/'V6_6_Final_Audit_Report.md').write_text(report)
print('V6.6 artifacts saved to:',OUTPUT_DIR)
print('Archive SHA-256:',archive_sha256)


## 9. Interactive CSV dashboard

Upload a new session CSV, set the score threshold and alert persistence, and inspect forecasts, risk scores and alert episodes. The risk threshold is adjustable because the score is not calibrated and the V5.9 diagnostics showed domain-dependent calibration problems.

In [ ]:
import gradio as gr

def dashboard_predict(file_path, threshold, persistence_k, cooldown_s):
    if file_path is None:
        raise gr.Error("Upload a CSV file first.")
    try:
        result, alerts, summary = run_inference(
            file_path, threshold=float(threshold),
            persistence_k=int(persistence_k), cooldown_s=float(cooldown_s)
        )
        stem = Path(file_path).stem
        pred_out = OUTPUT_DIR / f'{stem}_v6_predictions.csv'
        alert_out = OUTPUT_DIR / f'{stem}_v6_alerts.csv'
        result.to_csv(pred_out,index=False)
        alerts.to_csv(alert_out,index=False)
        return summary, result.tail(500), alerts, str(pred_out), (str(alert_out) if len(alerts) else None)
    except Exception as e:
        raise gr.Error(str(e))

with gr.Blocks(title="5G Network Intelligence V6.0") as demo:
    gr.Markdown("# 5G Network Intelligence — V6.6 Final Offline Prototype")
    gr.Markdown("Offline CSV inference • 5s/15s throughput forecast • KPI-derived risk score • persistent alert episodes")
    with gr.Row():
        input_file = gr.File(label="Upload one session CSV", file_types=[".csv"], type="filepath")
        threshold = gr.Slider(0.10,0.95,value=0.50,step=0.01,label="Risk-score threshold (not calibrated)")
    with gr.Row():
        persistence = gr.Slider(1,4,value=2,step=1,label="Consecutive positive rows")
        cooldown = gr.Slider(0,60,value=10,step=1,label="Alert cooldown (seconds)")
    run_btn = gr.Button("Run network analysis",variant="primary")
    summary_out = gr.JSON(label="Session summary")
    pred_out = gr.Dataframe(label="Latest scored rows (last 500)",interactive=False)
    alerts_out = gr.Dataframe(label="Confirmed alert episodes",interactive=False)
    with gr.Row():
        pred_file = gr.File(label="Download row-level predictions")
        alert_file = gr.File(label="Download alert episodes")
    run_btn.click(
        dashboard_predict,
        inputs=[input_file,threshold,persistence,cooldown],
        outputs=[summary_out,pred_out,alerts_out,pred_file,alert_file]
    )

demo.launch(share=False, debug=True)

## 10. Final acceptance checklist and honest limitations

**Functional checks**
- [ ] Training archive loads and session split is reconstructed.
- [ ] Forecast and warning models are saved to Google Drive.
- [ ] One-session smoke test completes without error.
- [ ] Dashboard accepts a CSV and returns row predictions plus alert episodes.
- [ ] Prediction and alert CSVs are saved under the V6.0 project folder.

**Not yet implemented**
- Live network telemetry / Kafka / gNB or O-RAN integration.
- Independently validated alert threshold or calibrated risk probabilities.
- Operator-confirmed incident labels and verified RCA.
- Multi-node topology, handover graph and production alert delivery.
- External validation on a new operator dataset.

Do not describe this as production-ready or as verified 5G fault prediction. It is a functional offline prototype built on client-side traces and KPI-derived proxy labels.